<a href="https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session2/BMEN35_Ex2_02_Logistic_regression_metrics_step_by_step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic regression: from scores to screening metrics

Logistic regression calculates a probability for the positive class; a threshold then turns that probability into a predicted class. In this walkthrough we write the model and metrics with NumPy rather than using a scikit-learn classifier.

We use the breast-cancer dataset included with scikit-learn as a teaching dataset. This is not a diagnostic tool and the results must not be used to make decisions about patients.

## 1. Import the tools

NumPy is used for the model calculations and Matplotlib for plots. We load the teaching dataset with scikit-learn, but we do not use a scikit-learn classifier. The observations are numerical arrays, so pandas is not needed in this walkthrough.

In [ ]:
# NumPy handles arrays and calculations; Matplotlib draws the plots.
import numpy as np
import matplotlib.pyplot as plt

# This function only loads a teaching dataset; it does not fit a model.
from sklearn.datasets import load_breast_cancer

# Use the same random split each time the notebook is run.
rng = np.random.default_rng(42)

## 2. Load the data and understand the labels

Each row describes a digitized breast-mass sample with 30 numerical measurements of cell nuclei. The original target uses 0 for malignant and 1 for benign. We recode it so that the positive class, label 1, means malignant.

It is important to write down this mapping. Later, sensitivity will mean the fraction of malignant observations that the model identifies.

In [ ]:
# Load the measurements and their feature names as NumPy arrays.
data = load_breast_cancer()
X_all = data.data.astype(float)
feature_names = list(data.feature_names)

# In the original target, 0=malignant and 1=benign.
# Recode it so our positive class is malignant (label 1).
y = (data.target == 0).astype(int)

print('Number of observations:', len(y))
print('Number of features:', X_all.shape[1])
print('Positive class (1): malignant')
print('Negative class (0): benign')
print('First five rows and first five features:')
print(np.round(X_all[:5, :5], 2))
print('First five feature names:', feature_names[:5])

## 3. Inspect and visualize the data

Before fitting a model, look at the class counts and a few feature summaries. Here we calculate the mean and standard deviation of the first five features using NumPy. Then we plot the number of observations in each class and two features.

The classifier will use all 30 features. The scatter plot is only a small window into the dataset, so it cannot show everything the model sees. These are real measurements in a public teaching dataset, but the notebook is not a clinical model.

In [ ]:
# Print simple summaries for the first five features.
for column in range(5):
    feature_values = X_all[:, column]
    print(feature_names[column])
    print('  mean:', round(np.mean(feature_values), 2))
    print('  standard deviation:', round(np.std(feature_values), 2))
    print('  minimum:', round(np.min(feature_values), 2))
    print('  maximum:', round(np.max(feature_values), 2))

# Count the two classes.
benign_count = np.sum(y == 0)
malignant_count = np.sum(y == 1)
print('Benign observations:', benign_count)
print('Malignant observations:', malignant_count)

# Plot the class counts so the class balance is easy to see.
plt.bar(['benign (0)', 'malignant (1)'], [benign_count, malignant_count],
        color=['steelblue', 'firebrick'], edgecolor='black')
plt.ylabel('Number of observations')
plt.title('Class balance in the teaching dataset')
plt.grid(axis='y', alpha=0.25)
plt.show()

# Find the two feature columns by their names.
radius_column = feature_names.index('mean radius')
texture_column = feature_names.index('mean texture')

# Draw a point for each observation, using colour to show its true class.
for class_value, label, color in [(0, 'benign', 'steelblue'), (1, 'malignant', 'firebrick')]:
    rows_from_this_class = y == class_value
    plt.scatter(X_all[rows_from_this_class, radius_column],
                X_all[rows_from_this_class, texture_column],
                label=label, color=color, alpha=0.65)

plt.xlabel('mean radius')
plt.ylabel('mean texture')
plt.title('Two features; colour shows the class')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### Stop and look

1. Do these two features completely separate the classes?
2. What information is left out of this plot?
3. Why is it important to state which class is positive?
4. What would be inappropriate to conclude from this teaching dataset?

## 4. Make training and test sets

We estimate parameters on training rows and reserve test rows for a final evaluation. We shuffle each class separately so that both sets contain benign and malignant observations. We set the test labels aside; they must not influence feature scaling or parameter updates.

In [ ]:
# Split each class separately, keeping its proportion similar in both sets.
benign=np.where(y==0)[0]; malignant=np.where(y==1)[0]
rng.shuffle(benign); rng.shuffle(malignant)
b_test=int(.25*len(benign)); m_test=int(.25*len(malignant))
test_idx=np.concatenate((benign[:b_test],malignant[:m_test]))
train_idx=np.concatenate((benign[b_test:],malignant[m_test:]))
X_train_raw=X_all[train_idx]; y_train=y[train_idx]
X_test_raw=X_all[test_idx]; y_test=y[test_idx]
print('Train rows:',len(y_train),' Test rows:',len(y_test))
print('Malignant fraction, train:',round(np.mean(y_train),3))
print('Malignant fraction, test:',round(np.mean(y_test),3))

## 5. Standardize using training data only

The features have different scales. For each feature, subtract its training mean and divide by its training standard deviation:

$$
z_{ij} = \frac{x_{ij}-\mu_j}{s_j}.
$$

Here, $x_{ij}$ is feature $j$ for observation $i$, $\mu_j$ is the mean of feature $j$ in the training data, and $s_j$ is its standard deviation. We calculate these values using training rows only, then apply the same values to both sets. Using test data to prepare the model would leak information from the held-out data.

In [ ]:
# 5. Standardize using training data only
means=np.mean(X_train_raw,axis=0)
stds=np.std(X_train_raw,axis=0)
stds[stds==0]=1.0
X_train=(X_train_raw-means)/stds
X_test=(X_test_raw-means)/stds
print('First five scaled training means:',np.round(X_train.mean(axis=0)[:5],3))
print('First five scaled training standard deviations:',np.round(X_train.std(axis=0)[:5],3))

## 6. From a linear score to a probability

For observation $i$, logistic regression first combines its feature values into a linear score:

$$
z_i = b + w_1x_{i1} + w_2x_{i2} + \cdots + w_px_{ip}.
$$

Here, $b$ is the intercept, $w_j$ is the weight for feature $j$, and $x_{ij}$ is feature $j$ for observation $i$. In vector notation this is $z_i=b+\mathbf{x}_i^T\mathbf{w}$.

The score is not yet a probability. The sigmoid function maps any score to a value between 0 and 1:

$$
p_i = \sigma(z_i) = \frac{1}{1+e^{-z_i}}.
$$

The output can be interpreted as the model's estimated probability that observation $i$ belongs to the positive class. For example, a probability of 0.8 means the model assigns an 80% probability to the positive class; it does not guarantee that the prediction is correct.

In [ ]:
# 6. From a linear score to a probability
z=np.linspace(-8,8,200)
probability=1/(1+np.exp(-z))
plt.plot(z,probability,color='darkorange',linewidth=2)
plt.axhline(.5,color='gray',linestyle='--'); plt.axvline(0,color='gray',linestyle=':')
plt.xlabel('linear score z'); plt.ylabel('sigmoid(z)')
plt.title('The sigmoid maps scores to probabilities'); plt.grid(alpha=.25); plt.show()

## 7. Write the model calculation

We start with all weights and the intercept equal to zero. The score is therefore zero for every observation, and the sigmoid maps each score to probability 0.5.

For a chosen threshold, we later assign class 1 if $p_i$ is at least the threshold; otherwise we assign class 0. The default threshold 0.5 is a starting point, not a rule that is always best.

In [ ]:
# Define the sigmoid function used to turn scores into probabilities.
def sigmoid(score):
    return 1 / (1 + np.exp(-score))  # The result is between 0 and 1.

# Start with an intercept of zero.
b = 0.0

# Start with one zero weight for each of the 30 input features.
w = np.zeros(X_train.shape[1])

# Calculate the linear scores for the first three training observations.
first_scores = b + X_train[:3] @ w

# Convert these scores into estimated probabilities.
first_probabilities = sigmoid(first_scores)

print('Scores:', first_scores)
print('Probabilities:', first_probabilities)
print('True labels:', y_train[:3])

## 8. Define the loss

Binary cross-entropy compares each probability $p_i$ with its known label $y_i$:

$$
L = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right].
$$

A confident correct prediction has a small loss; a confident incorrect prediction has a large loss. We clip probabilities very close to zero or one so the logarithm stays finite in computer arithmetic.

In [ ]:
# Define a function that calculates average binary cross-entropy.
def binary_cross_entropy(y_true, p):
    # Keep probabilities away from exactly 0 and 1 so log(p) remains finite.
    p = np.clip(p, 1e-12, 1 - 1e-12)

    # Calculate the loss for each row using its true label and predicted probability.
    row_loss = -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

    # Return the average loss across all observations.
    return np.mean(row_loss)

# At the start, every probability is 0.5 because all parameters are zero.
initial_p = sigmoid(b + X_train @ w)
print('Initial cross-entropy loss:', round(binary_cross_entropy(y_train, initial_p), 3))

## 9. Calculate gradients and take one update

Unlike linear regression, logistic regression does not have a normal equation that directly gives the parameter values minimizing cross-entropy. We therefore use gradient descent: calculate how the loss changes with the parameters, then update the parameters a little at a time.

For this loss, the gradients are:

$$
\frac{\partial L}{\partial b}=\frac{1}{n}\sum_i(p_i-y_i),\qquad
\frac{\partial L}{\partial \mathbf{w}}=\frac{1}{n}\mathbf{X}^T(\mathbf{p}-\mathbf{y}).
$$

The difference $p_i-y_i$ compares a predicted probability with its known label. Gradient descent subtracts each gradient from its parameter. We will calculate these quantities separately, then make one update so the change is visible.

In [ ]:
# Calculate current probabilities for all training observations.
p = sigmoid(b + X_train @ w)

# Positive values mean the current probability is above the observed label;
# negative values mean it is below the label.
error = p - y_train

# Average the errors to find the gradient for the intercept.
gradient_b = np.mean(error)

# Multiply each feature column by the errors and average to get its gradient.
gradient_w = (X_train.T @ error) / len(y_train)

# Choose how large each parameter update will be.
learning_rate = 0.1

# Subtract the gradient-scaled step to update the intercept and feature weights.
b_once = b - learning_rate * gradient_b
w_once = w - learning_rate * gradient_w

print('Intercept gradient:', round(gradient_b, 4))
print('First five weight gradients:', np.round(gradient_w[:5], 4))
print('Intercept after one update:', round(b_once, 4))
print('First five weights after one update:', np.round(w_once[:5], 4))

## 10. Train by repeating the same update

There is no direct analytical solution for the logistic-regression parameters in this example, so we repeat the gradient-descent update. Each step calculates probabilities, compares them with training labels, calculates the gradients, and updates the parameters. We store the loss after each update to see whether the training process is improving.

The learning rate controls the size of each update. The number of steps controls how many updates we take. If the learning rate is too large, the loss may jump around or increase; if it is too small, learning can be slow. This simple implementation is intended to show the calculations clearly.

In [ ]:
# Begin with a zero intercept and zero weight for every feature.
b = 0.0
w = np.zeros(X_train.shape[1])

# Set the step size and decide how many updates to make.
learning_rate = 0.1
number_of_steps = 3000
loss_history = []

for step in range(number_of_steps):
    # Calculate probabilities using the current parameter values.
    p = sigmoid(b + X_train @ w)

    # Compare probabilities with the known training labels.
    error = p - y_train

    # Calculate the gradients for the intercept and feature weights.
    gradient_b = np.mean(error)
    gradient_w = (X_train.T @ error) / len(y_train)

    # Update each parameter by moving against its gradient.
    b = b - learning_rate * gradient_b
    w = w - learning_rate * gradient_w

    # Save the loss after the update so we can plot it later.
    updated_p = sigmoid(b + X_train @ w)
    loss_history.append(binary_cross_entropy(y_train, updated_p))

print('Final training loss:', round(loss_history[-1], 3))
print('Intercept:', round(b, 3))
print('First five weights:', np.round(w[:5], 3))

Plotting the loss helps us see whether training reduced error on the training data. It does not tell us by itself how well the model works on new observations, which is why we reserved a test set.

In [ ]:
# 10. Train by repeating the same update
plt.plot(loss_history,color='darkorange')
plt.xlabel('Gradient-descent step'); plt.ylabel('Training cross-entropy')
plt.title('Loss during learning'); plt.grid(alpha=.25); plt.show()

## 11. Predict probabilities on the test set

The parameters are fixed. We calculate probabilities using the same training means and standard deviations. We do not use test labels to update the model. Probabilities can rank observations; a separate threshold changes them into class predictions.

In [ ]:
# Use the fitted parameters to calculate test-set probabilities.
malignant_probability = sigmoid(b + X_test @ w)

# Print a few labels and probabilities side by side.
for row in range(10):
    print('Test observation:', row)
    print('  true label:', y_test[row])
    print('  probability of malignant:', round(malignant_probability[row], 3))

# Plot probabilities separately for each true class.
for class_value, label, color in [(0, 'benign', 'steelblue'), (1, 'malignant', 'firebrick')]:
    class_rows = y_test == class_value  # True only for rows in this class.
    plt.hist(malignant_probability[class_rows], bins=15, alpha=0.6,
             label=label, color=color)

plt.xlabel('Estimated probability of malignant')
plt.ylabel('Test observations')
plt.title('Test probabilities grouped by true class')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

The histograms may overlap: some observations from different classes receive similar probabilities. A classifier can therefore make mistakes even though it returns probabilities rather than labels.

## 12. Threshold and confusion matrix

At threshold 0.5, a probability at least 0.5 becomes malignant (class 1); a smaller probability becomes benign (class 0). Changing the threshold changes the predicted labels, but it does not change the probabilities calculated by the model.

The confusion matrix below has true classes in rows and predicted classes in columns:

| | Predicted benign (0) | Predicted malignant (1) |
|---|---:|---:|
| **True benign (0)** | TN | FP |
| **True malignant (1)** | FN | TP |

- **TP — true positive:** malignant observation correctly predicted as malignant.
- **TN — true negative:** benign observation correctly predicted as benign.
- **FP — false positive:** benign observation incorrectly predicted as malignant.
- **FN — false negative:** malignant observation incorrectly predicted as benign.

The positive class is malignant in this notebook, so these names must be interpreted using that class mapping.

In [ ]:
# Choose a probability threshold for turning scores into class labels.
threshold = 0.5

# Probability at least the threshold becomes class 1 (malignant).
prediction = (malignant_probability >= threshold).astype(int)

# Count each kind of correct or incorrect prediction.
tn = np.sum((y_test == 0) & (prediction == 0))
fp = np.sum((y_test == 0) & (prediction == 1))
fn = np.sum((y_test == 1) & (prediction == 0))
tp = np.sum((y_test == 1) & (prediction == 1))

# Rows are true classes; columns are predicted classes.
confusion = np.array([[tn, fp], [fn, tp]])
print('Rows=true [benign, malignant], columns=predicted [benign, malignant]')
print(confusion)

## 13. Calculate and interpret the metrics

We calculate each measure from the four confusion-matrix counts. Their names and denominators tell us which question each one answers:

- **Accuracy:** $(TP+TN)/(TP+TN+FP+FN)$. What fraction of all observations did the model classify correctly?
- **Sensitivity**, also called **recall** or **true-positive rate (TPR):** $TP/(TP+FN)$. Of the malignant observations, what fraction did the model identify?
- **Specificity**, also called **true-negative rate (TNR):** $TN/(TN+FP)$. Of the benign observations, what fraction did the model correctly identify as benign?
- **Precision**, also called **positive predictive value (PPV):** $TP/(TP+FP)$. Of the observations predicted malignant, what fraction really were malignant?
- **F1 score:** $2\,\text{precision}\,\text{sensitivity}/(\text{precision}+\text{sensitivity})$. This is the harmonic mean of precision and sensitivity; it is high only when both are reasonably high.

Accuracy alone can be misleading when the classes have different sizes. A model can be correct on many observations by mostly predicting the more common class, while missing many positives. This is why we inspect the class balance and several metrics, and choose metrics that match the question being asked.

In [ ]:
# Accuracy: correct predictions divided by all test observations.
accuracy = (tp + tn) / len(y_test)

# Sensitivity: among true malignant cases, the fraction predicted malignant.
sensitivity = tp / (tp + fn)

# Specificity: among true benign cases, the fraction predicted benign.
specificity = tn / (tn + fp)

# Precision: among predictions of malignant, the fraction that are truly malignant.
precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

# F1 combines precision and sensitivity using their harmonic mean.
if precision + sensitivity > 0:
    f1 = 2 * precision * sensitivity / (precision + sensitivity)
else:
    f1 = np.nan

print('Accuracy:', round(accuracy, 3))
print('Sensitivity / recall / TPR:', round(sensitivity, 3))
print('Specificity / TNR:', round(specificity, 3))
print('Precision / PPV:', round(precision, 3))
print('F1 score:', round(f1, 3))

### Stop and interpret

1. What does a false negative mean here?
2. Which errors are described by sensitivity and specificity?
3. Could accuracy look high if a model mostly predicted the more common class? Compare with the class counts.
4. What does precision tell us that sensitivity does not?

## 14. Compare several thresholds

A lower threshold usually labels more observations positive. This can reduce false negatives while increasing false positives. A higher threshold usually does the opposite. The probabilities stay fixed; only the final class decisions change.

In [ ]:
# Store one row of counts and metrics for each threshold.
rows = []

for threshold in [0.20, 0.35, 0.50, 0.65, 0.80]:
    # Turn probabilities into labels using the current threshold.
    pred = (malignant_probability >= threshold).astype(int)

    # Count the four possible outcomes.
    tn = np.sum((y_test == 0) & (pred == 0))
    fp = np.sum((y_test == 0) & (pred == 1))
    fn = np.sum((y_test == 1) & (pred == 0))
    tp = np.sum((y_test == 1) & (pred == 1))

    # Calculate sensitivity and specificity for this threshold.
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    # Save the values so we can compare thresholds after the loop.
    rows.append([threshold, tn, fp, fn, tp, sensitivity, specificity])

# Print the column headings, then one line of values per threshold.
print('threshold   TN   FP   FN   TP   sensitivity   specificity')
for row in rows:
    print(f'{row[0]:9.2f} {row[1]:4d} {row[2]:4d} {row[3]:4d} {row[4]:4d}'
          f' {row[5]:12.3f} {row[6]:13.3f}')

### Compare the thresholds

1. Which threshold has highest sensitivity? Which has highest specificity?
2. As threshold increases, what happens to false negatives and false positives?
3. If missing a positive were considered especially costly in a hypothetical screening context, which direction might you explore? What trade-off comes with it?

A threshold must be chosen for a clearly stated goal and validated appropriately; one metric is not enough.

## 15. ROC curve and area

**ROC** stands for **receiver operating characteristic**. The ROC curve shows how the true-positive rate and false-positive rate change as we move the threshold. Here, $TPR=TP/(TP+FN)$ (sensitivity) and $FPR=FP/(FP+TN)$ (one minus specificity).

**AUC** stands for **area under the curve**. ROC-AUC summarizes how well the model ranks positive observations above negative observations across thresholds. It does not choose a threshold for us, and it does not tell us whether a particular threshold has acceptable errors.

The curve is represented by a finite set of points, one for each threshold. To estimate the area under it, we add the area between each neighboring pair of points. This is numerical integration using trapezoids. For neighboring points $i$ and $i+1$, the area is approximately

$$
(FPR_{i+1}-FPR_i)\,\frac{TPR_i+TPR_{i+1}}{2}.
$$

The width of each trapezoid is the change in FPR; its height is the average of the two TPR values. Adding all trapezoid areas approximates the integral of TPR with respect to FPR. More threshold points give a finer representation of the curve.

In [ ]:
# A threshold above every probability predicts all observations negative.
# A threshold below every probability predicts all observations positive.
thresholds = np.concatenate(([np.inf],
                             np.sort(np.unique(malignant_probability))[::-1],
                             [-np.inf]))

fpr = []  # False-positive rate at each threshold.
tpr = []  # True-positive rate at each threshold.

for threshold in thresholds:
    # Turn probabilities into predicted class labels at this threshold.
    pred = (malignant_probability >= threshold).astype(int)

    # Count true negatives, false positives, false negatives and true positives.
    tn = np.sum((y_test == 0) & (pred == 0))
    fp = np.sum((y_test == 0) & (pred == 1))
    fn = np.sum((y_test == 1) & (pred == 0))
    tp = np.sum((y_test == 1) & (pred == 1))

    # Calculate the rates that form one point on the ROC curve.
    fpr.append(fp / (fp + tn))
    tpr.append(tp / (tp + fn))

# Convert to arrays so that the points can be sorted by false-positive rate.
fpr = np.array(fpr)
tpr = np.array(tpr)
order = np.argsort(fpr)
fpr = fpr[order]
tpr = tpr[order]

# For each neighboring pair, calculate the width and average height.
widths = fpr[1:] - fpr[:-1]
average_heights = (tpr[1:] + tpr[:-1]) / 2
trapezoid_areas = widths * average_heights

# Sum the trapezoids: this approximates the integral under the ROC curve.
roc_auc = np.sum(trapezoid_areas)
print('ROC-AUC:', round(roc_auc, 3))

# Draw the ROC curve and a diagonal reference for a no-skill ranking.
plt.plot(fpr, tpr, color='darkorange', label=f'ROC-AUC={roc_auc:.3f}')
plt.plot([0, 1], [0, 1], '--', color='gray', label='no-skill reference')
plt.xlabel('False-positive rate (1 - specificity)')
plt.ylabel('True-positive rate (sensitivity)')
plt.title('ROC curve on held-out test set')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 16. Specific conclusions

1. What is the positive class? What does a false negative mean?
2. Compare sensitivity, specificity and the four counts at thresholds 0.20 and 0.80.
3. Why do probabilities stay the same while threshold metrics change?
4. What does ROC-AUC summarize, and what decision does it not make?
5. Why should the test set not be repeatedly used to choose a threshold?
6. What validation and expert review would be needed before considering a model for a health application?

The aim is to connect equations, code, plots and errors. This public dataset is used for teaching, not as a clinical decision system.